# Backtest Llama 3.1 8B: notícia versus sem notícia (pós-lançamento dos pesos)

Estratégia idêntica à do agente Luna, executada por **Llama 3.1 8B Instruct em 4 bits** na GPU do Colab, sobre o arquivo próprio `data/live_news/capture.jsonl` (16 emissores, capturado com `first_seen_at` desde dez/2023).

**Por que este período é mais defensável:** os pesos do Llama 3.1 foram publicados em 23/07/2024 com corte de conhecimento em dez/2023; as decisões começam em **ago/2024**, depois do lançamento, sobre eventos que os pesos não poderiam ter memorizado.

**Ablação pareada:** braço `news` (retornos + volatilidade + manchetes com atraso conservador de um pregão) versus braço `no_news` (mesmos números, sem manchetes), com controles de regra numérica e comprar-e-manter. Execução no **fechamento seguinte** à decisão, custos de 10 bps sobre o turnover.

**Limitações honestas:** os títulos são as versões atuais indexadas pelo Google News em 2026, não o texto publicado na época; a cobertura por janela é limitada a 100 itens pelo RSS; um resultado melhor no braço `news` não prova alfa futuro. É evidência exploratória, não recomendação.

In [ ]:
import sys, os
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

In [ ]:
import subprocess
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('Execute este notebook no Google Colab com GPU T4/L4') from exc
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Ative a GPU: Ambiente de execução → Alterar tipo de ambiente → T4/L4')
drive.mount('/content/drive')
REPO = Path('/content/latent_fusion')
if REPO.exists():
    if not (REPO / '.git').is_dir():
        raise RuntimeError(f'Pasta existente não é repositório Git: {REPO}')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/rbfcdog/latent_fusion.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.44,<5', 'accelerate', 'bitsandbytes', 'yfinance'], check=True)
{'gpu': torch.cuda.get_device_name(0), 'repo': str(REPO)}

In [ ]:
import json
import pandas as pd
from src.fusion.llama_news_backtest import load_news, load_prices_yfinance

DATA = Path('/content/drive/MyDrive/latent_fusion_data')
NEWS_PATH = DATA / 'live_news' / 'capture.jsonl'
PRICE_CACHE = DATA / 'live_news' / 'prices_2024_2026.parquet'
OUT_DIR = DATA / 'live_news' / 'llama_backtest_results'
if not NEWS_PATH.is_file():
    raise FileNotFoundError(f'Envie o arquivo capturado para o Drive: {NEWS_PATH}')
TICKERS = ['ADBE', 'AAPL', 'AMZN', 'BABA', 'COST', 'CVX', 'GILD', 'GOOGL',
           'JPM', 'META', 'MSFT', 'NKE', 'NVDA', 'PYPL', 'QCOM', 'TSLA']
if PRICE_CACHE.is_file():
    prices = pd.read_parquet(PRICE_CACHE)
else:
    prices = load_prices_yfinance(TICKERS, '2024-06-01', None)
    prices.to_parquet(PRICE_CACHE)
news = load_news(NEWS_PATH)
{'preços': {'sessões': len(prices), 'início': str(prices.index[0].date()), 'fim': str(prices.index[-1].date())},
 'manchetes': len(news), 'saída': str(OUT_DIR)}

## Modelo

O padrão usa o espelho **não restrito** `NousResearch/Meta-Llama-3.1-8B-Instruct` dos pesos oficiais do Llama 3.1 8B Instruct. Se você aceitou a licença da Meta no Hugging Face, troque `MODEL_ID` para `meta-llama/Meta-Llama-3.1-8B-Instruct` e autentique com `huggingface_hub.login()`. A quantização é de 4 bits via bitsandbytes; a decodificação é gulosa (determinística na prática).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'NousResearch/Meta-Llama-3.1-8B-Instruct'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map={'': 0}, torch_dtype=torch.bfloat16)
{'modelo': MODEL_ID, 'parâmetros': sum(p.numel() for p in model.parameters()) / 1e9}

In [ ]:
from tqdm.auto import tqdm
from src.fusion.llama_news_backtest import build_context, messages_for

PROGRESS = tqdm(total=None, unit='decisão')

def generate_fn(messages):
    PROGRESS.update(1)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=768, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

preview_date = prices.index[(prices.index >= pd.Timestamp('2024-08-05'))][0]
preview_context = build_context(prices, news, preview_date, TICKERS, 'news')
preview_messages = messages_for(preview_context)
{'caracteres_do_prompt': sum(len(m['content']) for m in preview_messages),
 'manchetes_no_prompt': sum(len(a['available_headlines']) for a in preview_context['assets'].values())}

## Execução

Decisões a cada 5 sessões de **01/08/2024** até o fim dos preços, nos dois braços. Cada resposta fica em cache (`decisions.jsonl`) indexada por hash do prompt e do modelo: desconexões do Colab retomam sem repetir inferências. Estimativa: ~100 datas × 2 braços; na L4/T4 4 bits, espere da ordem de 1 a 2,5 horas na primeira execução completa.

In [ ]:
from src.fusion.llama_news_backtest import run_llama_backtest

result = run_llama_backtest(prices, NEWS_PATH, OUT_DIR, tickers=TICKERS,
                            model_tag=MODEL_ID, generate_fn=generate_fn,
                            start='2024-08-01', end=str(prices.index[-1].date()),
                            every=5, cost_bps=10, max_headlines=8)
PROGRESS.close()
result['records'].head(8)

In [ ]:
summary = pd.DataFrame(result['summary']).T
summary.round(4)

In [ ]:
audit = result['records'].groupby('arm').agg(
    decisões=('decision_date', 'count'), json_válido=('parsed', 'mean'),
    manchetes_médias=('headlines', 'mean'), buys=('buy', 'sum'), sells=('sell', 'sum'))
audit.round(3)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.backtest.visualization import BG, PANEL, GOLD, GREEN, CYAN, WHITE

def _style(ax, title, ylabel):
    ax.set_facecolor(PANEL)
    ax.set_title(title, fontsize=16, color=WHITE)
    ax.set_ylabel(ylabel, fontsize=14, color=WHITE)
    ax.tick_params(colors=WHITE, labelsize=12)
    ax.grid(alpha=0.12)
    ax.legend(fontsize=14)

fig, ax = plt.subplots(figsize=(14, 5), facecolor=BG)
ax.plot(result['series']['news']['net'], color=GOLD, label='Llama + notícias (líquido)')
ax.plot(result['benchmark'], color=CYAN, label='Comprar e manter (16 ativos)')
ax.axhline(1.0, color=WHITE, alpha=0.2, linestyle='--')
_style(ax, 'Llama 3.1 8B com notícias versus comprar e manter', 'Patrimônio (base 1)')
fig.savefig(OUT_DIR / 'equity_llama_news_vs_bh.png', dpi=300, facecolor=BG, edgecolor='none')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5), facecolor=BG)
ax.plot(result['series']['news']['net'], color=GOLD, label='Com notícias')
ax.plot(result['series']['no_news']['net'], color=GREEN, label='Sem notícias')
ax.plot(result['series']['numeric']['net'], color=CYAN, label='Regra numérica')
ax.axhline(1.0, color=WHITE, alpha=0.2, linestyle='--')
_style(ax, 'Ablação pareada: valor incremental das notícias', 'Patrimônio líquido (base 1)')
fig.savefig(OUT_DIR / 'equity_news_ablation.png', dpi=300, facecolor=BG, edgecolor='none')
plt.show()

## Interpretação

A leitura correta é a **diferença pareada** `news` − `no_news` sobre retornos diários líquidos: ela isola o valor incremental do texto com o mesmo modelo, calendário, custos e regra de execução. O `t_stat_naive` ignora autocorrelação; trate-o como sinal exploratório, não como teste formal.

O que este notebook **não** prova: que os títulos atuais do Google News equivalem aos textos da época; que a cobertura de 100 itens por janela captura todo o fluxo de notícias; que um período favorecido se repetirá. Diferença positiva consistente nos dois braços é o resultado que justifica avançar para o registro prospectivo com o Luna e o diário `capture.jsonl` crescendo diariamente.